# Exercise 5: Checking a segmenter

*The Computational Phenotype · version 0.1.0*

> Run `MorphemeTokenizer(lang="tur")` on *yazıldılar* and *yazılamadılar* and on the Kazakh *жазылды* and *жазыла алмады*. Report which analysers were loaded. Where does the segmenter agree with your segmentation from Exercise 1, and where does it differ?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" pandas`. Download the language resources once with `turkicnlp.download(lang, processors=[...])`, as shown in the first code cell. Cells that fetch data from the web need internet access.

In [ ]:
import turkicnlp
from turkicnlp.processors.morpheme_tokenizer import MorphemeTokenizer

for lang in ["tur", "kaz"]:
    turkicnlp.download(lang, processors=["morph"])

reference = {"yazıldılar": "yaz-ıl-dı-lar", "yazılamadılar": "yaz-ıl-a-ma-dı-lar",
             "жазылды": "жаз-ыл-ды", "жазыла": "жаз-ыл-а", "алмады": "ал-ма-ды"}

for lang, words in [("tur", ["yazıldılar", "yazılamadılar"]), ("kaz", ["жазылды", "жазыла", "алмады"])]:
    seg = MorphemeTokenizer(lang=lang)
    seg.load()
    print(f"{lang}: neural={seg._neural_available} hfst={seg._hfst_available}")   # private flags in 0.5.0
    for word in words:
        r = seg.segment(word)
        auto = "-".join(m for m, _ in r.labeled)
        mark = "agrees" if auto == reference[word] else "differs"
        print(f"   {word:<14} auto {auto:<20} reference {reference[word]:<20} {mark}  {r.labeled} [{r.source}]")

## Worked solution (TurkicNLP 0.5.0, Apertium transducers only)

With only the Apertium transducers installed, the code prints `neural=False hfst=True` for both languages; report this with any result.

| Word | Segmenter | Reference | |
|---|---|---|---|
| *yazıldılar* | *yaz-ıl-dı-lar* (STEM, PASS, PST, 3PL) | *yaz-ıl-dı-lar* | agrees |
| *yazılamadılar* | *yaz-ıl-amadılar* (the last piece labelled `?`) | *yaz-ıl-a-ma-dı-lar* | differs |
| *жазылды* | *жаз-ыл-ды* (STEM, PASS, PST) | *жаз-ыл-ды* | agrees |
| *жазыла* | *жаз-ыл-а* (STEM, PASS, CVB) | *жаз-ыл-а* | agrees |
| *алмады* | *ал-ма-ды* (STEM, NEG, PST) | *ал-ма-ды* | agrees |

The segmenter handles the plain passive in both languages but fails on the Turkish inability form: it leaves *-amadılar* unsegmented and unlabelled, although the Apertium analysis itself (`<pass><abil><neg><ifi><p3><pl>`, Exercise 1) has all the information. Kazakh expresses the same meaning analytically, *жазыла алмады* (write-PASS-CVB take-NEG-PST): two words, each segmented correctly. The Turkish word and the Kazakh phrase therefore give different per-word counts for the same meaning.